In [1]:
import os
from pathlib import Path
from PyPDF2 import PdfReader

# ---------------------------
# CONFIGURATION
# ---------------------------
PDF_FOLDER = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag/data/pdfs")   # Folder containing PDFs
TXT_FOLDER = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag/data/txt")    # Folder containing TXT files
CHUNK_SIZE = 500                     # Approximate words per chunk

# ---------------------------
# FUNCTIONS
# ---------------------------

# Load PDF and extract text
def load_pdf(file_path):
    """Read a PDF file and return all text as a single string."""
    text = ""
    reader = PdfReader(file_path)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:  # only add if text exists
            text += page_text + "\n"
    return text

# Load TXT and extract text
def load_txt(file_path):
    """Read a TXT file and return all text as a single string."""
    with open(file_path, "r", encoding="utf-8") as f:
        return f.read()

# Split text into smaller chunks
def chunk_text(text, chunk_size=CHUNK_SIZE):
    """
    Split a long text into smaller chunks of 'chunk_size' words.
    """
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

# Main ingestion function
def ingest_documents():
    """
    Process all PDF and TXT files in the data folders.
    Returns a list of dictionaries containing:
    - doc_name: file name
    - chunk_id: automatic ID for each chunk
    - text: chunk text
    """
    all_chunks = []

    # Process PDFs
    for pdf_file in PDF_FOLDER.glob("*.pdf"):
        text = load_pdf(pdf_file)
        chunks = chunk_text(text)
        for idx, chunk in enumerate(chunks):
            all_chunks.append({
                "doc_name": pdf_file.name,
                "chunk_id": idx,  # automatic chunk ID
                "text": chunk
            })

    # Process TXT files
    for txt_file in TXT_FOLDER.glob("*.txt"):
        text = load_txt(txt_file)
        chunks = chunk_text(text)
        for idx, chunk in enumerate(chunks):
            all_chunks.append({
                "doc_name": txt_file.name,
                "chunk_id": idx,  # automatic chunk ID
                "text": chunk
            })

    print(f"[INFO] Total chunks created: {len(all_chunks)}")
    return all_chunks

# ---------------------------
# MAIN TEST
# ---------------------------
if __name__ == "__main__":
    chunks = ingest_documents()
    print("Sample chunk from first document:")
    print(f"Document: {chunks[3]['doc_name']}, Chunk ID: {chunks[1]['chunk_id']}")
    print(chunks[1]['text'][:500] + "...")

[INFO] Total chunks created: 20
Sample chunk from first document:
Document: reinforcement_learning.pdf, Chunk ID: 1
jobs ● Time-Sharing Systems: Allow multiple users to share resources ● Distributed Systems: Manage multiple interconnected computers ● Real-Time Systems: Provide immediate responses for critical tasks Each type is designed for specific use cases and performance requirements. 11. Virtualization Virtualization allows multiple virtual machines to run on a single physical system. Benefits: ● Better resource utilization ● Isolation between environments ● Scalability Technologies like hypervisors enab...


In [2]:
import sys
from pathlib import Path
import pickle
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# Add project root to Python path
PROJECT_ROOT = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag")
sys.path.append(str(PROJECT_ROOT))

# Import your ingestion function
from ingestion_pipeline.ingestion import ingest_documents
# ---------------------------
# CONFIGURATION
# ---------------------------
EMBEDDING_MODEL = "all-MiniLM-L6-v2"  # SentenceTransformers model
EMBEDDINGS_FOLDER = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag/embeddings")
EMBEDDINGS_FOLDER.mkdir(exist_ok=True)  # create folder if not exists

# ---------------------------
# LOAD CHUNKS
# ---------------------------
print("[INFO] Loading documents and splitting into chunks...")
chunks = ingest_documents()  # returns list of dicts with doc_name, chunk_id, text

# ---------------------------
# INITIALIZE EMBEDDING MODEL
# ---------------------------
print(f"[INFO] Loading embedding model: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)

# ---------------------------
# CREATE EMBEDDINGS
# ---------------------------
embeddings_data = []  # will store embeddings + metadata

print("[INFO] Generating embeddings...")
for chunk in tqdm(chunks):
    vector = model.encode(chunk["text"])
    embeddings_data.append({
        "doc_name": chunk["doc_name"],
        "chunk_id": chunk.get("chunk_id"),  # optional if you are using it
        "text": chunk["text"],
        "embedding": vector
    })

# ---------------------------
# SAVE EMBEDDINGS
# ---------------------------
output_file = EMBEDDINGS_FOLDER / "all_embeddings.pkl"
with open(output_file, "wb") as f:
    pickle.dump(embeddings_data, f)

print(f"[INFO] Embeddings saved to {output_file}")
print(f"[INFO] Total embeddings created: {len(embeddings_data)}")

/Users/malavjoshi/Desktop/RAG_Projects/local_rag/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[INFO] Loading documents and splitting into chunks...
[INFO] Total chunks created: 20
[INFO] Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8397.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Generating embeddings...


100%|██████████| 20/20 [00:00<00:00, 32.65it/s]

[INFO] Embeddings saved to /Users/malavjoshi/Desktop/RAG_Projects/local_rag/embeddings/all_embeddings.pkl
[INFO] Total embeddings created: 20


In [3]:
import pickle
from pathlib import Path
import faiss
from tqdm import tqdm

# ---------------------------
# CONFIGURATION
# ---------------------------
EMBEDDINGS_FILE = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag/embeddings/all_embeddings.pkl")  # embeddings
VECTOR_STORE_FOLDER = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag/vector_store")
VECTOR_STORE_FOLDER.mkdir(exist_ok=True)

FAISS_INDEX_FILE = VECTOR_STORE_FOLDER / "faiss_index.index"
METADATA_FILE = VECTOR_STORE_FOLDER / "metadata.pkl"

# ---------------------------
# LOAD EMBEDDINGS
# ---------------------------
print("[INFO] Loading embeddings...")
with open(EMBEDDINGS_FILE, "rb") as f:
    embeddings_data = pickle.load(f)

print(f"[INFO] Total chunks to index: {len(embeddings_data)}")

# ---------------------------
# EXTRACT VECTORS AND METADATA
# ---------------------------
vectors = []
metadata = []

for item in tqdm(embeddings_data):
    vectors.append(item["embedding"])  # the vector
    metadata.append({
        "doc_name": item["doc_name"],
        "chunk_id": item.get("chunk_id"),
        "text": item["text"]
    })

# Convert to float32 numpy array for FAISS
import numpy as np
vectors = np.array(vectors).astype("float32")
dimension = vectors.shape[1]
print(f"[INFO] Embedding dimension: {dimension}")

# ---------------------------
# BUILD FAISS INDEX
# ---------------------------
print("[INFO] Building FAISS index...")
index = faiss.IndexFlatL2(dimension)  # L2 distance
index.add(vectors)
print(f"[INFO] Total vectors indexed: {index.ntotal}")

# ---------------------------
# SAVE INDEX AND METADATA
# ---------------------------
faiss.write_index(index, str(FAISS_INDEX_FILE))
with open(METADATA_FILE, "wb") as f:
    pickle.dump(metadata, f)

print(f"[INFO] FAISS index saved to: {FAISS_INDEX_FILE}")
print(f"[INFO] Metadata saved to: {METADATA_FILE}")

[INFO] Loading embeddings...
[INFO] Total chunks to index: 20


100%|██████████| 20/20 [00:00<00:00, 409200.39it/s]

[INFO] Embedding dimension: 384
[INFO] Building FAISS index...
[INFO] Total vectors indexed: 20
[INFO] FAISS index saved to: /Users/malavjoshi/Desktop/RAG_Projects/local_rag/vector_store/faiss_index.index
[INFO] Metadata saved to: /Users/malavjoshi/Desktop/RAG_Projects/local_rag/vector_store/metadata.pkl


In [4]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

# ---------------------------
# PATH SETUP
# ---------------------------
PROJECT_ROOT = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag")
VECTOR_STORE = PROJECT_ROOT / "vector_store"

FAISS_INDEX_FILE = VECTOR_STORE / "faiss_index.index"
METADATA_FILE = VECTOR_STORE / "metadata.pkl"
# ---------------------------
# LOAD INDEX + METADATA
# ---------------------------
index = faiss.read_index(str(FAISS_INDEX_FILE))

with open(METADATA_FILE, "rb") as f:
    metadata = pickle.load(f)

# ---------------------------
# LOAD EMBEDDING MODEL
# ---------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

# ---------------------------
# QUERY -- 1
# ---------------------------
query_1 = "What is the difference between deep learning and machine learning?"

query_vector_1 = model.encode([query_1]).astype("float32")

# ---------------------------
# SEARCH TOP-K
# ---------------------------
k = 3
distances, indices = index.search(query_vector_1, k)

print("\n[QUERY 1]:", query_1)
print("\nTop Results:\n")

for i, idx in enumerate(indices[0]):
    result = metadata[idx]
    print(f"Rank {i+1}")
    print(f"Document: {result['doc_name']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Text: {result['text'][:200]}...\n")

# ---------------------------
# QUERY -- 2
# ---------------------------
query_2 = "What is Operating System?"

query_vector_2 = model.encode([query_2]).astype("float32")

# ---------------------------
# SEARCH TOP-K
# ---------------------------
k = 3
distances, indices = index.search(query_vector_2, k)

print("\n[QUERY 2]:", query_2)
print("\nTop Results:\n")

for i, idx in enumerate(indices[0]):
    result = metadata[idx]
    print(f"Rank {i+1}")
    print(f"Document: {result['doc_name']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Text: {result['text'][:200]}...\n")

# ---------------------------
# QUERY -- 3    
# ---------------------------
query_3 = "What is rag and vector databases?"

query_vector_3 = model.encode([query_3]).astype("float32")

# ---------------------------
# SEARCH TOP-K
# ---------------------------
k = 3
distances, indices = index.search(query_vector_3, k)

print("\n[QUERY 3]:", query_3)
print("\nTop Results:\n")

for i, idx in enumerate(indices[0]):
    result = metadata[idx]
    print(f"Rank {i+1}")
    print(f"Document: {result['doc_name']}")
    print(f"Chunk ID: {result['chunk_id']}")
    print(f"Text: {result['text'][:200]}...\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7812.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[QUERY 1]: What is the difference between deep learning and machine learning?

Top Results:

Rank 1
Document: deep_learning.pdf
Chunk ID: 0
Text: Deep Learning Basics 1. Introduction to Deep Learning Deep learning is a specialized branch of machine learning that uses neural networks with multiple layers to model complex patterns in data. It has...

Rank 2
Document: machine_learning.pdf
Chunk ID: 0
Text: Machine Learning Introduction 1. Introduction to Machine Learning Machine learning is a branch of artificial intelligence that focuses on enabling computers to learn patterns from data without being e...

Rank 3
Document: deep_learning.pdf
Chunk ID: 1
Text: modeling, machine translation, and speech recognition. 8. Applications of Deep Learning Deep learning has a wide range of applications: ● Computer Vision: Image classification, object detection, facia...


[QUERY 2]: What is Operating System?

Top Results:

Rank 1
Document: operating_system.pdf
Chunk ID: 0
Text: Operating Systems 1.

In [5]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

class Retriever:
    """
    Retrieval module for RAG pipeline.
    Converts query to embeddings, searches FAISS, and returns top-k chunks.
    """

    def __init__(self, index_path: str, metadata_path: str, model_name: str = "all-MiniLM-L6-v2"):
        # Load FAISS index
        self.index = faiss.read_index(index_path)

        # Load metadata
        with open(metadata_path, "rb") as f:
            self.metadata = pickle.load(f)

        # Load embedding model
        self.model = SentenceTransformer(model_name)

    def search(self, query: str, top_k: int = 3):
        """
        Search the FAISS index for the query and return top-k chunks with metadata.
        """
        query_vector = self.model.encode([query]).astype("float32")
        distances, indices = self.index.search(query_vector, top_k)

        results = []
        for idx in indices[0]:
            result = self.metadata[idx]
            results.append({
                "doc_name": result["doc_name"],
                "chunk_id": result["chunk_id"],
                "text": result["text"]
            })
        return results


# ---------------------------
# Quick Test
# ---------------------------
if __name__ == "__main__":
    retriever = Retriever(
        index_path="/Users/malavjoshi/Desktop/RAG_Projects/local_rag/vector_store/faiss_index.index",
        metadata_path="/Users/malavjoshi/Desktop/RAG_Projects/local_rag/vector_store/metadata.pkl"
    )

    query = "What is deep learning?"
    results = retriever.search(query, top_k=3)

    print(f"\n[QUERY]: {query}\n")
    for i, r in enumerate(results):
        print(f"Rank {i+1}")
        print(f"Document: {r['doc_name']}")
        print(f"Chunk ID: {r['chunk_id']}")
        print(f"Text: {r['text'][:300]}...\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8691.90it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[QUERY]: What is deep learning?

Rank 1
Document: deep_learning.pdf
Chunk ID: 0
Text: Deep Learning Basics 1. Introduction to Deep Learning Deep learning is a specialized branch of machine learning that uses neural networks with multiple layers to model complex patterns in data. It has gained attention due to its ability to handle large amounts of unstructured data, such as images, t...

Rank 2
Document: deep_learning.pdf
Chunk ID: 1
Text: modeling, machine translation, and speech recognition. 8. Applications of Deep Learning Deep learning has a wide range of applications: ● Computer Vision: Image classification, object detection, facial recognition ● Natural Language Processing: Sentiment analysis, translation, summarization ● Speech...

Rank 3
Document: reinforcement_learning.pdf
Chunk ID: 1
Text: learning algorithms. It learns the optimal action-value function using the update rule: Q(s, a) = Q(s, a) + α [r + γ max Q(s’, a’) − Q(s, a)] Where: ● α (alpha): Learning rate ● γ (gamma):

In [21]:
# generator.py
# ---------------------------
# Imports
# ---------------------------
import sys
from pathlib import Path

# Project root
PROJECT_ROOT = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag")
sys.path.append(str(PROJECT_ROOT))

# Your retriever module
from retrieval.retriever import Retriever

# Ollama SDK
import ollama

# ---------------------------
# Generator Class
# ---------------------------
class Generator:
    def __init__(self, model_name="mistral:latest", index_path=None, metadata_path=None):
        """
        Initialize the Generator with Ollama model and Retriever.
        """
        if index_path is None or metadata_path is None:
            raise ValueError("You must provide index_path and metadata_path for the Retriever.")

        # Load Retriever
        self.retriever = Retriever(index_path=index_path, metadata_path=metadata_path)
        print(f"[INFO] Retriever loaded with index: {index_path}")

        # Ollama model
        self.model_name = model_name
        print(f"[INFO] Generator initialized with Ollama model: {self.model_name}")

    def generate(self, query, top_k=5):
        """
        Generate an answer using retrieved chunks + Ollama LLM
        """
        # Step 1: Retrieve top-k relevant chunks
        docs = self.retriever.search(query, top_k=top_k)
        context = "\n".join([doc["text"] for doc in docs])

        # Step 2: Construct prompt
        prompt = f"Answer the following query based on context:\n\nContext:\n{context}\n\nQuery: {query}\nAnswer:"

        # Step 3: Generate response using Ollama LLM
        response = ollama.generate(model=self.model_name, prompt=prompt)

        # Step 4: Return the actual generated text
        return response.response
        
        

# ---------------------------
# Example Usage
# ---------------------------
if __name__ == "__main__":
    # Update these paths according to your project
    INDEX_PATH = PROJECT_ROOT / "vector_store/faiss_index.index"
    METADATA_PATH = PROJECT_ROOT / "vector_store/metadata.pkl"

    # Initialize generator
    generator = Generator(
        model_name="mistral:latest",
        index_path=INDEX_PATH,
        metadata_path=METADATA_PATH
    )

    # Example query
    query = "What is internet?"
    answer = generator.generate(query)

    print("\n=== Generated Answer ===")
    print(answer.replace(".",".\n"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8004.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[INFO] Retriever loaded with index: /Users/malavjoshi/Desktop/RAG_Projects/local_rag/vector_store/faiss_index.index
[INFO] Generator initialized with Ollama model: mistral:latest

=== Generated Answer ===
 The Internet is a global network of interconnected computers that forms the backbone of digital communication.
 It defines how data is transmitted, routed, and received efficiently across networks worldwide.
 Devices like computers, smartphones, IoT devices, etc.
, request and receive data from servers via routers and switches.
 Understanding Internet architecture is essential for network engineers, software developers, and cybersecurity professionals.
 The internet relies on standardized protocols and a layered model that allows devices of different types and capabilities to communicate seamlessly.



In [ ]:
import pickle
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu

# ---------------------------
# Project Paths
# ---------------------------
PROJECT_ROOT = Path("/Users/malavjoshi/Desktop/RAG_Projects/local_rag")
METADATA_PATH = PROJECT_ROOT / "vector_store/metadata.pkl"
INDEX_PATH = PROJECT_ROOT / "vector_store/faiss_index.index"

# ---------------------------
# Load Metadata
# ---------------------------
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

# ---------------------------
# Initialize Embedding Model
# ---------------------------
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ---------------------------
# Evaluation Functions
# ---------------------------
def retrieval_accuracy(query, retrieved_chunks, top_k=5):
    """
    Evaluate retrieval: Compute cosine similarity of query vs retrieved chunks.
    """
    query_vec = embedding_model.encode([query])
    scores = []
    for chunk in retrieved_chunks:
        chunk_vec = embedding_model.encode([chunk["text"]])
        sim = cosine_similarity(query_vec, chunk_vec)[0][0]
        scores.append(sim)
    avg_score = sum(scores) / len(scores)
    print(f"[Retrieval] Average Cosine Similarity (Top-{top_k}): {avg_score:.4f}")
    return avg_score

def generation_quality(predicted_answer, reference_answer):
    """
    Evaluate generated answer using ROUGE and BLEU
    """
    # ROUGE
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(reference_answer, predicted_answer)

    # BLEU
    reference_tokens = [reference_answer.split()]
    predicted_tokens = predicted_answer.split()
    bleu_score = sentence_bleu(reference_tokens, predicted_tokens)

    print("\n[Generation] Evaluation Metrics")
    print("ROUGE-1:", rouge_scores['rouge1'].fmeasure)
    print("ROUGE-L:", rouge_scores['rougeL'].fmeasure)
    print("BLEU:", bleu_score)

    return rouge_scores, bleu_score

# ---------------------------
# Quick Test
# ---------------------------
if __name__ == "__main__":
    from retrieval.retriever import Retriever
    from generation.generator import Generator

    # Example query and expected answer
    query = "What is retrieval augmented generation?"
    expected_answer = ("Retrieval-Augmented Generation (RAG) is a technique where a language "
                       "model retrieves relevant documents from a knowledge base to generate "
                       "more accurate and factual responses.")

    # Load retriever & generator
    retriever = Retriever(index_path=INDEX_PATH, metadata_path=METADATA_PATH)
    generator = Generator(model_name="mistral:latest",
                          index_path=INDEX_PATH,
                          metadata_path=METADATA_PATH)

    # Step 1: Retrieval evaluation
    top_chunks = retriever.search(query, top_k=5)
    retrieval_accuracy(query, top_chunks, top_k=5)

    # Step 2: Generation evaluation
    generated_answer = generator.generate(query)
    generation_quality(generated_answer, expected_answer)